# Contoso PMO KB MCP server

Synthetic database of projects, people, tasks, meetings, risks, and documents (MoMs, team chats, emails, plans), exposed via a multi-tool Azure Functions MCP server (tools like `get_overdue_tasks`, `search_lessons`, and `search_documents`). A Foundry agent acting as a conversational project assistant consumes this MCP server to help the described personas query project status, surface relevant lessons learned from past projects etc etc.

An illustration of this use case, but also others where there's more complexity including custom integration with other systems are required (e.g., SAP, task management, messaging).

---

Builds a **Model Context Protocol (MCP) server** backed by a JSON knowledge base and deploys it on Azure Functions (Flex Consumption, Python). A Foundry agent on the **admin project** (`project-admin-{suffix}`) is then connected to the server via `PromptAgentDefinition` using the versioned-agent API, giving project managers and cross-functional teams a conversational AI assistant powered by 37 structured tools.

This notebook is the **canonical creator** of the `contoso-pmo-agent`. Downstream notebooks - [`08-05-02-contoso-pmo-agent-queries.ipynb`](08-05-02-contoso-pmo-agent-queries.ipynb) and the full [`08-06-agent-offline-evaluation/`](../08-06-agent-offline-evaluation/) series - look this agent up by name. Do not recreate it elsewhere.

```
assets/contoso-pmo-dataset/←── JSON knowledge base (registry + documents)
    │
    ▼
contoso-pmo-mcp/           ←── Azure Functions app
  function_app.py         37 mcpToolTrigger wrappers
  kb.py                   Business logic (read + write)
  host.json               Standard GA extension bundle [4, 5)
    │
    ▼ SSE endpoint /runtime/webhooks/mcp/sse
    │
PromptAgentDefinition     ←── tools=[{type:'mcp', require_approval:'never'}]
    │
    ▼ project_client.agents.create_version on project-admin-{suffix}
Foundry Agent (versioned) ←── contoso-pmo-agent v1, v2, …
```

## Prerequisites

1. **Hub deployment complete** - the admin project (`project-admin-{suffix}` on `aif-core-{suffix}`) must already exist. The endpoint is derived deterministically from the subscription ID.
2. **Python environment** - run `uv sync` from the repo root; select the `.venv` kernel.
3. **Azure CLI** - `az login` with an account that has `Contributor` + `User Access Administrator` on the target subscription, plus permissions to create agents on the admin project.
4. **Required RBAC roles on the function-app storage** (assigned automatically by this notebook):
   - `Storage Blob Data Owner` - deployment package access
   - `Storage Queue Data Contributor` - MCP SSE transport (queue creation)
   - `Storage Table Data Contributor` - host metadata
5. **`.env`** - only `CHAT_MODEL` is required (e.g. `CHAT_MODEL=gpt-4.1-mini`). `AZURE_SUBSCRIPTION_ID` is optional; if unset, the active `az` subscription is used.

Optional `.env` overrides:
- `CONTOSO_PMO_MCP_RESOURCE_GROUP` (default: `rg-foundry-contoso-pmo-mcp`)
- `CONTOSO_PMO_MCP_LOCATION` (default: `swedencentral`)
- `CONTOSO_PMO_FUNC_APP_NAME` (default: derived as `func-contoso-pmo-mcp-{md5(sub-id+rg)[:6]}`)

## Imports and configuration

Load `.env` from the repo root and configure resource names.

In [1]:
import hashlib
import json
import os
import subprocess
import time
import zipfile
from pathlib import Path

from dotenv import load_dotenv

repo_root = Path(
    subprocess.run('git rev-parse --show-toplevel', shell=True, capture_output=True, text=True).stdout.strip()
)
load_dotenv(repo_root / '.env', override=True)

CHAT_MODEL = os.environ.get('CHAT_MODEL', 'gpt-4.1-mini')

# Subscription + admin project endpoint - same suffix-derivation pattern as 08-06-01 and 08-07-06.
SUBSCRIPTION_ID = (
    os.environ.get('AZURE_SUBSCRIPTION_ID')
    or subprocess.run('az account show --query id -o tsv',
                      shell=True, capture_output=True, text=True).stdout.strip()
)
SUFFIX           = hashlib.sha256((SUBSCRIPTION_ID + 'v2').encode()).hexdigest()[:6]
PROJECT_ENDPOINT = f'https://aif-core-{SUFFIX}.services.ai.azure.com/api/projects/project-admin-{SUFFIX}'

# MCP function-app resource group + deterministic naming (md5 of sub-id+rg).
CONTOSO_PMO_MCP_RG  = os.environ.get('CONTOSO_PMO_MCP_RESOURCE_GROUP', 'rg-foundry-contoso-pmo-mcp')
CONTOSO_PMO_MCP_LOC = os.environ.get('CONTOSO_PMO_MCP_LOCATION', 'swedencentral')
_mcp_suffix    = hashlib.md5(f'{SUBSCRIPTION_ID}-{CONTOSO_PMO_MCP_RG}'.encode()).hexdigest()[:6]
STORAGE_NAME   = f'stcontosopmomcp{_mcp_suffix}'
FUNC_APP_NAME  = os.environ.get('CONTOSO_PMO_FUNC_APP_NAME') or f'func-contoso-pmo-mcp-{_mcp_suffix}'

AGENT_NAME = 'contoso-pmo-agent'

SOURCE_DIR = repo_root / '08-agents' / '08-05-contoso-pmo-mcp' / 'contoso-pmo-mcp'
ZIP_PATH   = repo_root / '08-agents' / '08-05-contoso-pmo-mcp' / 'contoso-pmo-mcp.zip'

print(f'Subscription     : {SUBSCRIPTION_ID}')
print(f'Admin endpoint   : {PROJECT_ENDPOINT}')
print(f'Resource group   : {CONTOSO_PMO_MCP_RG}')
print(f'Location         : {CONTOSO_PMO_MCP_LOC}')
print(f'Storage account  : {STORAGE_NAME}')
print(f'Function app     : {FUNC_APP_NAME}')
print(f'Agent name       : {AGENT_NAME}')
print(f'Chat model       : {CHAT_MODEL}')
print(f'Source dir       : {SOURCE_DIR}')

Subscription     : 00000000-0000-0000-0000-000000000000
Admin endpoint   : https://aif-core-c2676f.services.ai.azure.com/api/projects/project-admin-c2676f
Resource group   : rg-foundry-contoso-pmo-mcp
Location         : swedencentral
Storage account  : stcontosopmomcpd55074
Function app     : func-contoso-pmo-mcp-d55074
Agent name       : contoso-pmo-agent
Chat model       : gpt-4.1-mini
Source dir       : <repo-root>/08-agents/08-05-contoso-pmo-mcp/contoso-pmo-mcp


## Authentication

Uses `DefaultAzureCredential` for both the Foundry SDK and Azure management calls. Run `az login` first.

In [2]:
from azure.identity import DefaultAzureCredential

foundry_credential = DefaultAzureCredential()
print('DefaultAzureCredential initialised.')

DefaultAzureCredential initialised.


---
## Phase 1: Build the MCP server

Writes the Azure Functions source files into `contoso-pmo-mcp/` and bundles the `assets/data/` knowledge base. The `function_app.py` and `kb.py` files are already present in `contoso-pmo-mcp/`; this phase writes configuration files and copies the data.

In [3]:
import json as _json
import shutil

SOURCE_DIR.mkdir(exist_ok=True)

# ── host.json (standard GA extension bundle) ───────────────────────────────────────────────
(SOURCE_DIR / 'host.json').write_text(_json.dumps({
    "version": "2.0",
    "extensionBundle": {
        "id": "Microsoft.Azure.Functions.ExtensionBundle",
        "version": "[4.0.0, 5.0.0)"
    }
}, indent=2) + '\n')

# ── requirements.txt ─────────────────────────────────────────────────────────────────────────
(SOURCE_DIR / 'requirements.txt').write_text('azure-functions\n')

# ── Verify function_app.py and kb.py are present ───────────────────────────────────────────────
for _name in ('function_app.py', 'kb.py'):
    _p = SOURCE_DIR / _name
    assert _p.exists(), f'{_name} missing from {SOURCE_DIR}'
    print(f'  {_name}: {_p.stat().st_size:,} bytes')

# ── Bundle assets/contoso-pmo-dataset/ into SOURCE_DIR/data/ ───────────────────────────────────────
_data_dest = SOURCE_DIR / 'data'
if _data_dest.exists():
    shutil.rmtree(_data_dest)
shutil.copytree(str(repo_root / 'assets' / 'contoso-pmo-dataset'), _data_dest)

print(f'\nSource directory: {SOURCE_DIR.resolve()}')
print(f'  host.json       : ready (standard GA extension bundle)')
print(f'  requirements.txt: ready')
print(f'  data/           : {len(list(_data_dest.rglob("*.json")))} JSON files bundled')

  function_app.py: 36,372 bytes
  kb.py: 23,697 bytes

Source directory: <repo-root>/08-agents/08-05-contoso-pmo-mcp/contoso-pmo-mcp
  host.json       : ready (standard GA extension bundle)
  requirements.txt: ready
  data/           : 21 JSON files bundled


---
## Phase 2: Deploy to Azure

Provisions a dedicated resource group with a Flex Consumption function app using managed identity for storage (no shared keys). Assigns the three storage roles required for MCP SSE transport and sets `DATA_DIR=data` so the function reads the bundled knowledge base.

In [4]:
def _run(cmd, label):
    """Run an az CLI command via subprocess. Prints ✓ or ✗ and returns the result."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode == 0:
        print(f'  ✓ {label}')
    else:
        print(f'  ✗ {label}')
        err = (result.stderr or result.stdout).strip()
        if err:
            print(f'    {err[:400]}')
    return result


STORAGE_SCOPE = (
    f'/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{CONTOSO_PMO_MCP_RG}'
    f'/providers/Microsoft.Storage/storageAccounts/{STORAGE_NAME}'
)

print('Creating resource group...')
_run(
    f'az group create --name "{CONTOSO_PMO_MCP_RG}" --location "{CONTOSO_PMO_MCP_LOC}" -o none',
    f"resource group '{CONTOSO_PMO_MCP_RG}'")

# `--public-network-access Enabled` is required: recent Azure subscription defaults
# disable public endpoints on new storage accounts, which breaks both the
# `az storage container create --auth-mode login` call below AND
# `az functionapp create` (which refuses to bind a function app to a
# network-restricted storage account without VNet integration).
# `--allow-shared-key-access false` keeps the secure key-less posture (managed
# identity is used for AzureWebJobsStorage); only the *network* layer is opened.
print('Creating storage account...')
_run(
    f'az storage account create -g "{CONTOSO_PMO_MCP_RG}" -n "{STORAGE_NAME}" '
    f'-l "{CONTOSO_PMO_MCP_LOC}" --sku Standard_LRS '
    f'--allow-shared-key-access false --public-network-access Enabled -o none',
    f"storage account '{STORAGE_NAME}'")

print('Assigning current user role for container creation...')
_current_user = subprocess.run(
    'az ad signed-in-user show --query id -o tsv',
    shell=True, capture_output=True, text=True).stdout.strip()
_run(
    f'az role assignment create --assignee "{_current_user}" '
    f'--role "Storage Blob Data Contributor" --scope "{STORAGE_SCOPE}" -o none',
    'Storage Blob Data Contributor → current user')
print('  … waiting 15 s for RBAC to propagate...')
time.sleep(15)

_run(
    f'az storage container create --account-name "{STORAGE_NAME}" '
    f'--name deployments --auth-mode login -o none',
    'deployments blob container')

print(f"Creating function app '{FUNC_APP_NAME}' (~2 min)...")
_run(
    f'az functionapp create -g "{CONTOSO_PMO_MCP_RG}" -n "{FUNC_APP_NAME}" '
    f'--storage-account "{STORAGE_NAME}" --runtime python --runtime-version 3.11 '
    f'--flexconsumption-location "{CONTOSO_PMO_MCP_LOC}" '
    f'--deployment-storage-container-name deployments '
    f'--deployment-storage-auth-type SystemAssignedIdentity -o none',
    f"function app '{FUNC_APP_NAME}'")
print('  … waiting 30 s for managed identity to initialize...')
time.sleep(30)

print('Assigning storage roles to managed identity...')
_PRINCIPAL = subprocess.run(
    f'az functionapp identity show -g "{CONTOSO_PMO_MCP_RG}" -n "{FUNC_APP_NAME}" '
    f'--query principalId -o tsv',
    shell=True, capture_output=True, text=True).stdout.strip()

if not _PRINCIPAL:
    print('  ✗ managed identity not found - function app may not have provisioned')
else:
    print(f'  ✓ managed identity principal: {_PRINCIPAL}')
    for _role in [
        'Storage Blob Data Owner',
        'Storage Queue Data Contributor',
        'Storage Table Data Contributor',
    ]:
        _run(
            f'az role assignment create --assignee-object-id "{_PRINCIPAL}" '
            f'--assignee-principal-type ServicePrincipal '
            f'--role "{_role}" --scope "{STORAGE_SCOPE}" -o none',
            _role)

print('Configuring identity-based storage connection...')
_run(
    f'az functionapp config appsettings delete -g "{CONTOSO_PMO_MCP_RG}" '
    f'-n "{FUNC_APP_NAME}" --setting-names AzureWebJobsStorage -o none',
    'removed key-based AzureWebJobsStorage')
_run(
    f'az functionapp config appsettings set -g "{CONTOSO_PMO_MCP_RG}" '
    f'-n "{FUNC_APP_NAME}" '
    f'--settings AzureWebJobsStorage__accountName={STORAGE_NAME} '
    f'AzureWebJobsStorage__credential=managedidentity -o none',
    'set managed identity storage connection')
_run(
    f'az functionapp config appsettings set -g "{CONTOSO_PMO_MCP_RG}" '
    f'-n "{FUNC_APP_NAME}" --settings DATA_DIR=data -o none',
    'DATA_DIR=data')

Creating resource group...
  ✓ resource group 'rg-foundry-contoso-pmo-mcp'
Creating storage account...
  ✓ storage account 'stcontosopmomcpd55074'
Assigning current user role for container creation...
  ✓ Storage Blob Data Contributor → current user
  … waiting 15 s for RBAC to propagate...
  ✓ deployments blob container
Creating function app 'func-contoso-pmo-mcp-d55074' (~2 min)...
  ✗ function app 'func-contoso-pmo-mcp-d55074'
    ERROR: Cannot change the site func-contoso-pmo-mcp-d55074 to the App Service Plan ASP-rgfoundrycontosopmomcp-e6cf due to hosting constraints.
  … waiting 30 s for managed identity to initialize...
Assigning storage roles to managed identity...
  ✓ managed identity principal: 00000000-0000-0000-0000-000000000000
  ✓ Storage Blob Data Owner
  ✓ Storage Queue Data Contributor
  ✓ Storage Table Data Contributor
Configuring identity-based storage connection...
  ✓ removed key-based AzureWebJobsStorage
  ✓ set managed identity storage connection
  ✓ DATA_DIR=dat

CompletedProcess(args='az functionapp config appsettings set -g "rg-foundry-contoso-pmo-mcp" -n "func-contoso-pmo-mcp-d55074" --settings DATA_DIR=data -o none', returncode=0, stdout='', stderr='')

---
## Phase 3: Package and deploy code

Zips the `contoso-pmo-mcp/` source directory - including the bundled knowledge base - into `contoso-pmo-mcp.zip` and deploys it to the Azure Functions app via `az functionapp deployment source config-zip`. Re-run this phase alone whenever `function_app.py`, `kb.py`, or the data files change.

In [5]:
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for _fp in SOURCE_DIR.rglob('*'):
        if _fp.is_file():
            zf.write(_fp, _fp.relative_to(SOURCE_DIR))

print(f'Zip archive: {ZIP_PATH} ({ZIP_PATH.stat().st_size:,} bytes)')
print('Deploying code...')
_deploy = subprocess.run(
    f'az functionapp deployment source config-zip '
    f'-g "{CONTOSO_PMO_MCP_RG}" -n "{FUNC_APP_NAME}" --src "{ZIP_PATH}"',
    shell=True, capture_output=True, text=True)
if _deploy.returncode == 0 or '202' in (_deploy.stdout + _deploy.stderr):
    print('  ✓ code deployed')
else:
    print(f'  ✗ code deployment failed')
    print(f'    {(_deploy.stdout + _deploy.stderr)[:400]}')

Zip archive: <repo-root>/08-agents/08-05-contoso-pmo-mcp/contoso-pmo-mcp.zip (39,660 bytes)
Deploying code...
  ✓ code deployed


---
## Phase 4: Retrieve function URL and key

The SSE endpoint Foundry connects to requires the `mcp_extension` system key. This key is generated by the MCP extension after first load - the cell retries up to 6 times (120 seconds total) while the extension initialises.

In [6]:
FUNC_BASE_URL = f'https://{FUNC_APP_NAME}.azurewebsites.net'

print('Retrieving MCP system key (mcp_extension)...')
MCP_KEY = ''
for _attempt in range(6):
    _r = subprocess.run(
        f'az functionapp keys list -g "{CONTOSO_PMO_MCP_RG}" -n "{FUNC_APP_NAME}" '
        f'--query "systemKeys.mcp_extension" -o tsv',
        shell=True, capture_output=True, text=True)
    MCP_KEY = _r.stdout.strip()
    if MCP_KEY and MCP_KEY != 'None':
        print('  ✓ MCP system key (mcp_extension) retrieved')
        break
    print(f'  … not ready yet, waiting 20 s (attempt {_attempt + 1}/6)...')
    time.sleep(20)

if not MCP_KEY or MCP_KEY == 'None':
    print('  ✗ MCP system key not found - check function app logs')
    MCP_KEY = ''

MCP_SSE_URL = f'{FUNC_BASE_URL}/runtime/webhooks/mcp/sse?code={MCP_KEY}'

print(f'Function base URL : {FUNC_BASE_URL}')
print(f'MCP SSE endpoint  : {FUNC_BASE_URL}/runtime/webhooks/mcp/sse?code=<key>')

Retrieving MCP system key (mcp_extension)...
  ✓ MCP system key (mcp_extension) retrieved
Function base URL : https://func-contoso-pmo-mcp-d55074.azurewebsites.net
MCP SSE endpoint  : https://func-contoso-pmo-mcp-d55074.azurewebsites.net/runtime/webhooks/mcp/sse?code=<key>


---
## Phase 4.5: Smoke-test the MCP server

Verifies the function app is reachable and the MCP SSE endpoint accepts the `mcp_extension` system key **before** the agent is created. Catches deployment problems at deploy time rather than surfacing them later as opaque `tool_server_error / 504` failures during agent runs.

If this cell raises:

- **Function-app HEAD timeout** - the function app isn't responding at all. Check `az functionapp show -g <rg> -n <name> --query state` and the function-app logs in the Azure portal.
- **MCP SSE 504 / read timeout** - the worker is up but the MCP extension isn't producing the SSE stream within 30 s. Most often a Flex Consumption cold start; re-run the cell. If persistent, recreate the function app.
- **HTTP 401 on the SSE URL** - the `mcp_extension` system key is stale; re-run Phase 4 to refresh `MCP_KEY`.
- **HTTP 4xx with body mentioning the extension** - the function-app code or extension bundle didn't deploy cleanly; re-run Phase 3.

In [7]:
import urllib.request
import urllib.error
import socket

print('Smoke-testing MCP server reachability...')

# 1. Function-app root - confirms the worker is up.
try:
    _req = urllib.request.Request(FUNC_BASE_URL, method='HEAD')
    with urllib.request.urlopen(_req, timeout=60) as _r:
        print(f'  ✓ Function app root responding (HTTP {_r.status})')
except urllib.error.HTTPError as _e:
    # 401/403/404 on the bare root is fine - confirms the app is alive
    print(f'  ✓ Function app root responding (HTTP {_e.code})')
except (socket.timeout, urllib.error.URLError) as _e:
    raise RuntimeError(
        f"Function app '{FUNC_APP_NAME}' did not respond within 60 s. "
        f"It may be in a bad state (cold-start failure, networking issue, or stopped).\n"
        f"  Check: az functionapp show -g {CONTOSO_PMO_MCP_RG} -n {FUNC_APP_NAME} --query state -o tsv\n"
        f"  Error: {_e}"
    ) from _e

# 2. MCP SSE endpoint - confirms the mcp_extension is reachable with the system key.
print('  Opening MCP SSE stream...')
try:
    _req = urllib.request.Request(MCP_SSE_URL, headers={'Accept': 'text/event-stream'})
    with urllib.request.urlopen(_req, timeout=30) as _r:
        if _r.status != 200:
            raise RuntimeError(f'MCP SSE returned HTTP {_r.status}')
        # Read a short chunk to confirm the stream produces data.
        # MCP servers emit an `endpoint` event immediately on connection.
        _chunk = _r.read(256)
        if not _chunk:
            raise RuntimeError('MCP SSE opened but produced no data - extension may not be ready')
        print(f'  ✓ MCP SSE stream alive ({len(_chunk)} bytes received in first chunk)')
except urllib.error.HTTPError as _e:
    _body = _e.read(400) if hasattr(_e, 'read') else b''
    raise RuntimeError(
        f"MCP SSE endpoint returned HTTP {_e.code}. "
        f"Possible causes: stale mcp_extension key, extension not yet loaded, "
        f"or function-app code mismatch.\n"
        f"  URL : {FUNC_BASE_URL}/runtime/webhooks/mcp/sse?code=<key>\n"
        f"  Body: {_body!r}"
    ) from _e
except (socket.timeout, urllib.error.URLError) as _e:
    raise RuntimeError(
        f"MCP SSE endpoint timed out or unreachable.\n"
        f"This is exactly what causes 'tool_server_error / 504' failures during agent runs.\n"
        f"Most common root causes:\n"
        f"  - Function-app cold start exceeded the 30 s timeout - re-run this cell\n"
        f"  - Function-app deployment is in a bad state - recreate {FUNC_APP_NAME}\n"
        f"  - Storage-account network restrictions blocking the worker\n"
        f"  Error: {_e}"
    ) from _e

print('\n  ✓ MCP server smoke test passed - safe to create the agent')

Smoke-testing MCP server reachability...
  ✓ Function app root responding (HTTP 200)
  Opening MCP SSE stream...
  ✓ MCP SSE stream alive (256 bytes received in first chunk)

  ✓ MCP server smoke test passed - safe to create the agent


---
## Phase 5: Create the Foundry agent on the admin project

Creates the canonical `contoso-pmo-agent` on `project-admin-{suffix}` via the
versioned-agent API (`project_client.agents.create_version` +
`PromptAgentDefinition`). The MCP tool is attached as a tool-definition dict
with `require_approval='never'` baked in - no per-run `ToolSet` plumbing is
needed in [`08-05-02-contoso-pmo-agent-queries.ipynb`](08-05-02-contoso-pmo-agent-queries.ipynb).

**Why admin project:** Foundry's model-graded evaluators (used in
[`08-06-02-quality-evaluators.ipynb`](../08-06-agent-offline-evaluation/08-06-02-quality-evaluators.ipynb) and
[`08-06-03-agent-evaluators.ipynb`](../08-06-agent-offline-evaluation/08-06-03-agent-evaluators.ipynb))
need a model deployment they can reach directly. Spoke accounts have zero
deployments by policy; the admin project hosts the centrally-managed
deployments on `aif-core-{suffix}`.

**08-06 evaluation chain note:** the offline-evaluation lab was originally
written against the Classic Assistants API and captures `thread_id` / `run_id`
into `test_data.jsonl`. The Responses API (used by [`08-05-02`](08-05-02-contoso-pmo-agent-queries.ipynb))
produces a `response.id` instead. If you re-run [`08-06-01`](../08-06-agent-offline-evaluation/08-06-01-setup-and-test-data.ipynb)
to regenerate test data against this agent, expect the run-capture flow to need
updating; that work is tracked separately from this lab.

**Idempotency:** uses a reuse-or-create pattern via `list_versions`. Re-running
this notebook returns the latest existing version rather than appending another.
Set `force_new_version = True` in the cell below to force a new version (e.g.
after editing the instructions).


In [8]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.core.exceptions import ResourceNotFoundError

project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=foundry_credential)

MCP_SSE_URL = f'{FUNC_BASE_URL}/runtime/webhooks/mcp/sse?code={MCP_KEY}'

_INSTRUCTIONS = (
    'You are the Contoso PMO knowledge base assistant. You help project managers '
    'and cross-functional team members in a consumer product launch environment.\n\n'
    'KNOWLEDGE BASE TOOLS\n'
    'You have 37 MCP tools covering projects, people, meetings, tasks, risks, '
    'documents, and distribution lists. Always use these tools to read and write '
    'data - never guess or fabricate IDs, names, or dates.\n\n'
    'APPROVAL WORKFLOW\n'
    'Tasks and documents are created unapproved (approved=false). Use approve_task '
    'and approve_document explicitly when asked to approve items.\n\n'
    'GATE FRAMEWORK\n'
    'Projects follow the BLAST gate framework: G0 (concept) → G1 (feasibility) → '
    'G2 (development) → G3 (validation) → G4 (launch readiness) → G5 (launch) → '
    'G6 (post-launch review) → G7 (closure).\n\n'
    'LESSONS LEARNED\n'
    'When discussing project risks or planning for an upcoming gate, proactively '
    'search lessons learned (search_lessons, search_risk_patterns) to surface '
    'relevant past experience from completed projects.'
)

force_new_version = False  # set True to bump a new version on re-run

# Reuse-or-create: list_versions raises ResourceNotFoundError when the agent
# does not yet exist; treat that as "create the first version".
try:
    existing_versions = list(project_client.agents.list_versions(agent_name=AGENT_NAME))
except ResourceNotFoundError:
    existing_versions = []

if existing_versions and not force_new_version:
    agent = existing_versions[0]
    print(f"Reusing existing agent '{agent.name}' v{agent.version}")
else:
    agent = project_client.agents.create_version(
        agent_name=AGENT_NAME,
        definition=PromptAgentDefinition(
            model=CHAT_MODEL,
            instructions=_INSTRUCTIONS,
            tools=[
                {
                    'type': 'mcp',
                    'server_label': 'contoso_pmo_kb',
                    'server_url': MCP_SSE_URL,
                    'require_approval': 'never',
                },
            ],
        ),
        description='Contoso PMO KB assistant backed by a 37-tool Azure Functions MCP server.',
    )
    print(f"Created agent '{agent.name}' v{agent.version}")

print(f'MCP server label : contoso_pmo_kb')
print(f'MCP SSE endpoint : {FUNC_BASE_URL}/runtime/webhooks/mcp/sse?code=<key>')


Created agent 'contoso-pmo-agent' v1
MCP server label : contoso_pmo_kb
MCP SSE endpoint : https://func-contoso-pmo-mcp-d55074.azurewebsites.net/runtime/webhooks/mcp/sse?code=<key>
